# Pgen model: run and analyze

This notebook analyzes a trained sequence-to-`log10(pgen_1mm)` model.

It covers:
- loading the saved model and metadata
- reproducing the train/val/test split
- comparing predicted vs real `log10(pgen_1mm)`
- inspecting residuals and worst examples
- benchmarking inference time for several batch sizes

## 1. Define inputs

In [ ]:
from pathlib import Path

root = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()

# Example TRB paths; update these for the chain/run you want to inspect.
airr_path = Path('/projects/immunestatus/vdjrearm/pgen_1mm_background_100k/trb_background_100k_pgen.tsv')
model_dir = Path('/projects/immunestatus/vdjrearm/irrmcodec/pgen_background_100k/trb')

airr_path, model_dir

In [ ]:
import sys
sys.path.insert(0, str(root))

## 2. Imports

In [ ]:
import json
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from torch.utils.data import DataLoader

from irrm_codec.dataio import normalize_locus_name, read_airr_table
from irrm_codec.datasets import PgenDataset, collate_pgen
from irrm_codec.pgen_model import PgenModel
from irrm_codec.tokenization import strip_gaps
from irrm_codec.utils import split_indices

sns.set_theme(context='notebook', style='whitegrid')

## 3. Load saved metadata

In [ ]:
data_stats = json.loads((model_dir / 'data_stats.json').read_text())
history = json.loads((model_dir / 'history.json').read_text())
test_metrics = json.loads((model_dir / 'test_metrics.json').read_text())

data_stats, test_metrics

## 4. Rebuild the test split

In [ ]:
target_col = data_stats.get('target_column', 'log10_pgen_1mm')
clone_id_col = data_stats.get('clone_id_column', 'clone_id')
locus = data_stats.get('locus')
seed = int(data_stats.get('seed', 42))
train_fraction = float(data_stats.get('train_fraction', 0.8))
val_fraction = float(data_stats.get('val_fraction', 0.1))
max_len = int(data_stats.get('max_len', 40))

df = read_airr_table(airr_path, clone_id_col=clone_id_col)
if locus is not None:
    locus_norm = normalize_locus_name(locus)
    locus_series = df['locus'].astype(str).str.strip().str.lower().map(normalize_locus_name)
    df = df.loc[locus_series == locus_norm].reset_index(drop=True)

target_array = pd.to_numeric(df[target_col], errors='coerce').to_numpy(dtype=np.float32)
train_idx, val_idx, test_idx = split_indices(
    len(df),
    train_fraction=train_fraction,
    val_fraction=val_fraction,
    seed=seed,
)

test_df = df.iloc[test_idx].reset_index(drop=True).copy()
test_target = target_array[test_idx]
test_df['target_log10_pgen_1mm'] = test_target

len(df), len(test_df)

## 5. Load the trained model

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
checkpoint = torch.load(model_dir / 'best.pt', map_location=device)
extra = checkpoint.get('extra', {})

model = PgenModel(
    max_len=int(extra.get('max_len', max_len)),
    hidden_dim=int(extra.get('hidden_dim', 192)),
    mlp_dim=int(extra.get('mlp_dim', 512)),
    mlp_hidden_dim=int(extra.get('mlp_hidden_dim', 1024)),
    dropout=float(extra.get('dropout', 0.2)),
    dilations=tuple(int(part.strip()) for part in str(extra.get('dilations', '1,2,4,8')).split(',') if part.strip()),
    encoder_type=extra.get('encoder_type', 'plain_conv'),
).to(device)
model.load_state_dict(checkpoint['model_state'])
model.eval()

device, extra

## 6. Predict on the test split

In [ ]:
eval_batch_size = 1024
test_loader = DataLoader(
    PgenDataset(test_df, test_target, max_len=max_len),
    batch_size=eval_batch_size,
    shuffle=False,
    num_workers=0,
    collate_fn=collate_pgen,
)

pred_batches = []
with torch.no_grad():
    for tokens, mask, target, _lengths in test_loader:
        tokens = tokens.to(device)
        mask = mask.to(device)
        pred = model(tokens, mask).detach().cpu().numpy()
        pred_batches.append(pred)

test_df['pred_log10_pgen_1mm'] = np.concatenate(pred_batches)
test_df['residual'] = test_df['pred_log10_pgen_1mm'] - test_df['target_log10_pgen_1mm']
test_df['abs_error'] = test_df['residual'].abs()
test_df['seq_length'] = test_df['junction_aa'].astype(str).map(lambda x: len(strip_gaps(x.replace('-', '').upper())))

test_df.head()

## 7. Summary metrics

In [ ]:
y_true = test_df['target_log10_pgen_1mm'].to_numpy(dtype=np.float64)
y_pred = test_df['pred_log10_pgen_1mm'].to_numpy(dtype=np.float64)
residual = y_pred - y_true

rmse = float(np.sqrt(np.mean((y_pred - y_true) ** 2)))
mae = float(np.mean(np.abs(y_pred - y_true)))
bias = float(np.mean(residual))
pearson = float(np.corrcoef(y_true, y_pred)[0, 1]) if len(y_true) > 1 else float('nan')
spearman = float(
    np.corrcoef(
        pd.Series(y_true).rank(method='average').to_numpy(),
        pd.Series(y_pred).rank(method='average').to_numpy(),
    )[0, 1]
) if len(y_true) > 1 else float('nan')
ss_res = float(np.sum((y_true - y_pred) ** 2))
ss_tot = float(np.sum((y_true - np.mean(y_true)) ** 2))
r2 = 1.0 - ss_res / ss_tot if ss_tot > 0 else float('nan')

summary = pd.DataFrame([
    {
        'rmse': rmse,
        'mae': mae,
        'bias': bias,
        'pearson': pearson,
        'spearman': spearman,
        'r2': r2,
        'n_test': len(test_df),
    }
])
summary

## 8. Error plots

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 11))

sns.scatterplot(
    data=test_df.sample(min(len(test_df), 20000), random_state=42),
    x='target_log10_pgen_1mm',
    y='pred_log10_pgen_1mm',
    s=12,
    alpha=0.35,
    ax=axes[0, 0],
)
xy_min = min(test_df['target_log10_pgen_1mm'].min(), test_df['pred_log10_pgen_1mm'].min())
xy_max = max(test_df['target_log10_pgen_1mm'].max(), test_df['pred_log10_pgen_1mm'].max())
axes[0, 0].plot([xy_min, xy_max], [xy_min, xy_max], '--', color='black', linewidth=1)
axes[0, 0].set_title('Predicted vs real log10(pgen_1mm)')

sns.histplot(test_df['residual'], bins=80, kde=True, ax=axes[0, 1])
axes[0, 1].set_title('Residual distribution')
axes[0, 1].set_xlabel('predicted - real')

sns.scatterplot(
    data=test_df.sample(min(len(test_df), 20000), random_state=42),
    x='target_log10_pgen_1mm',
    y='residual',
    s=12,
    alpha=0.35,
    ax=axes[1, 0],
)
axes[1, 0].axhline(0.0, linestyle='--', color='black', linewidth=1)
axes[1, 0].set_title('Residual vs real target')

length_error = test_df.groupby('seq_length', as_index=False)['abs_error'].mean()
sns.lineplot(data=length_error, x='seq_length', y='abs_error', marker='o', ax=axes[1, 1])
axes[1, 1].set_title('Mean absolute error by sequence length')

plt.tight_layout()

## 9. Largest errors

In [ ]:
worst = test_df.sort_values('abs_error', ascending=False).head(20)
worst[[clone_id_col, 'junction_aa', 'locus', 'target_log10_pgen_1mm', 'pred_log10_pgen_1mm', 'residual', 'abs_error']]

## 10. Inference speed benchmark

In [ ]:
def benchmark_inference(model, df, device, max_len, batch_sizes=(1, 16, 64, 256, 1024), num_repeats=5):
    records = []
    for batch_size in batch_sizes:
        loader = DataLoader(
            PgenDataset(df, np.zeros(len(df), dtype=np.float32), max_len=max_len),
            batch_size=batch_size,
            shuffle=False,
            num_workers=0,
            collate_fn=collate_pgen,
        )

        with torch.no_grad():
            for tokens, mask, _target, _lengths in loader:
                tokens = tokens.to(device)
                mask = mask.to(device)
                _ = model(tokens, mask)
                break
            if device.type == 'cuda':
                torch.cuda.synchronize()

        elapsed = []
        for _ in range(num_repeats):
            start = time.perf_counter()
            with torch.no_grad():
                for tokens, mask, _target, _lengths in loader:
                    tokens = tokens.to(device)
                    mask = mask.to(device)
                    _ = model(tokens, mask)
                if device.type == 'cuda':
                    torch.cuda.synchronize()
            elapsed.append(time.perf_counter() - start)

        mean_seconds = float(np.mean(elapsed))
        records.append(
            {
                'batch_size': batch_size,
                'num_sequences': int(len(df)),
                'mean_seconds': mean_seconds,
                'std_seconds': float(np.std(elapsed)),
                'seq_per_second': float(len(df) / mean_seconds),
                'ms_per_sequence': float(mean_seconds * 1000.0 / len(df)),
            }
        )
    return pd.DataFrame(records)


benchmark_df = benchmark_inference(
    model=model,
    df=test_df[['junction_aa']].assign(locus=test_df['locus']).assign(v_call=test_df['v_call']).assign(j_call=test_df['j_call']),
    device=device,
    max_len=max_len,
    batch_sizes=(1, 16, 64, 256, 1024),
    num_repeats=5,
)
benchmark_df

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

sns.barplot(data=benchmark_df, x='batch_size', y='seq_per_second', ax=axes[0])
axes[0].set_title('Inference throughput')
axes[0].set_ylabel('sequences / second')

sns.barplot(data=benchmark_df, x='batch_size', y='ms_per_sequence', ax=axes[1])
axes[1].set_title('Latency per sequence')
axes[1].set_ylabel('ms / sequence')

plt.tight_layout()